In [0]:

%sql
CREATE TABLE IF NOT EXISTS DimCourse (
    code_module STRING NOT NULL,
    CONSTRAINT pk_dimcourse PRIMARY KEY (code_module)
) USING DELTA;

-- ----------------------------------------------------------
-- DIM: DimDate (grain: calendar date)
-- ----------------------------------------------------------
CREATE TABLE IF NOT EXISTS DimDate (
    date            INT NOT NULL,
    relative_week   INT,
    course_phase    STRING,
    CONSTRAINT pk_dimdate PRIMARY KEY (date)
) USING DELTA;

-- ----------------------------------------------------------
-- DIM: DimStudent (grain: student)
-- ----------------------------------------------------------
CREATE TABLE IF NOT EXISTS DimStudent (
    id_student          INT NOT NULL,
    final_result        STRING,
    date_registration   INT,
    date_unregistration INT,
    is_withdrawn        BOOLEAN,
    CONSTRAINT pk_dimstudent PRIMARY KEY (id_student)
) USING DELTA;

-- ----------------------------------------------------------
-- DIM: DimModulePresentation (grain: module + presentation)
-- ----------------------------------------------------------
CREATE TABLE IF NOT EXISTS DimModulePresentation (
    code_module              STRING NOT NULL,
    code_presentation        STRING NOT NULL,
    module_presentation_length INT,
    CONSTRAINT pk_dimmodpres PRIMARY KEY (code_module, code_presentation),
    CONSTRAINT fk_dimmodpres_course FOREIGN KEY (code_module)
        REFERENCES DimCourse (code_module)
) USING DELTA;

-- ----------------------------------------------------------
-- DIM: DimDemographics (grain: student + module + presentation)
-- ----------------------------------------------------------
CREATE TABLE IF NOT EXISTS DimDemographics (
    id_student              INT NOT NULL,
    code_module             STRING NOT NULL,
    code_presentation       STRING NOT NULL,
    gender                  STRING,
    region                  STRING,
    highest_education       STRING,
    imd_band                STRING,
    age_band                STRING,
    num_of_previous_attempts INT,
    studied_credits         INT,
    disability              STRING,
    CONSTRAINT pk_dimdemo PRIMARY KEY (id_student, code_module, code_presentation),
    CONSTRAINT fk_dimdemo_student FOREIGN KEY (id_student)
        REFERENCES DimStudent (id_student),
    CONSTRAINT fk_dimdemo_modpres FOREIGN KEY (code_module, code_presentation)
        REFERENCES DimModulePresentation (code_module, code_presentation)
) USING DELTA;

-- ----------------------------------------------------------
-- FACT: FactVLEInteractions
-- ----------------------------------------------------------
CREATE TABLE IF NOT EXISTS FactVLEInteractions (
    id_student          INT NOT NULL,
    code_module         STRING NOT NULL,
    code_presentation   STRING NOT NULL,
    date                INT NOT NULL,
    id_site             INT NOT NULL,
    activity_type       STRING,
    sum_click           INT,
    CONSTRAINT pk_factvle PRIMARY KEY (id_student, code_module, code_presentation, date, id_site),
    CONSTRAINT fk_factvle_student FOREIGN KEY (id_student)
        REFERENCES DimStudent (id_student),
    CONSTRAINT fk_factvle_modpres FOREIGN KEY (code_module, code_presentation)
        REFERENCES DimModulePresentation (code_module, code_presentation),
    CONSTRAINT fk_factvle_date FOREIGN KEY (date)
        REFERENCES DimDate (date)
) USING DELTA;

-- ----------------------------------------------------------
-- FACT: FactAssessments
-- ----------------------------------------------------------
CREATE TABLE IF NOT EXISTS FactAssessments (
    id_student          INT NOT NULL,
    code_module         STRING NOT NULL,
    code_presentation   STRING NOT NULL,
    date                INT NOT NULL,
    id_assessment       INT NOT NULL,
    assessment_type     STRING,
    weight              DECIMAL(5,2),
    score               DECIMAL(5,2),
    date_submitted      INT,
    submission_delay    INT,
    is_banked           BOOLEAN,
    CONSTRAINT pk_factassess PRIMARY KEY (id_student, code_module, code_presentation, date, id_assessment),
    CONSTRAINT fk_factassess_student FOREIGN KEY (id_student)
        REFERENCES DimStudent (id_student),
    CONSTRAINT fk_factassess_modpres FOREIGN KEY (code_module, code_presentation)
        REFERENCES DimModulePresentation (code_module, code_presentation),
    CONSTRAINT fk_factassess_date FOREIGN KEY (date)
        REFERENCES DimDate (date)
) USING DELTA;